In [ ]:
import os, shutil, json, re
from pathlib import Path
import pandas as pd
from datetime import datetime
from collections import Counter

# Define BIDS root and derivatives roots
dataset = "zurich-bids"
bids_root = Path(f"/data/project/sleep_ENIGMA_glymphatic/enigma-sd/{dataset}")
derivatives_roots = [bids_root / "derivatives/fmriprep", bids_root / "derivatives/freesurfer"]

# Read participant data
participants = pd.read_csv(bids_root / "participants.tsv", sep="\t")
has_first_session = "first_session" in participants.columns

# Session label maps
session_map = { # Update this based on old/new labels
    "ses-bl": "ses-T1BL",
    "ses-sd": "ses-T2SD",
    "ses-rc": "ses-T3RC"
}
crossover_map = { # Update this if the study used a crossover design
    "ses-bl": "ses-T1BL",
    "ses-sd": "ses-T2SD",
    "ses-rc": "ses-T3RC"
}
control_map = { # Update this if the study had a control group
    "ses-bl": "ses-T1BL",
    "ses-sd": "ses-T2SD",
    "ses-rc": "ses-T3RC"
}

def design_from_sessions(participant_id):
    """ntnu-bids only: no first_session column, so infer design from the session folders."""
    sessions = {p.name for p in (bids_root / participant_id).glob("ses-*")}
    if "ses-T2" in sessions and "ses-T4" not in sessions:
        return crossover_map
    if "ses-T4" in sessions and "ses-T2" not in sessions:
        return session_map
    print(f"WARNING {participant_id}: ambiguous sessions {sorted(sessions)}, using session_map")
    return session_map

def build_session_map(participant_id, first_session=None, group=None):
    if pd.notna(group) and int(group) < 3:
        return control_map
    if dataset == "ntnu-bids":
        return design_from_sessions(participant_id)
    if pd.isna(first_session) or first_session in ("BL", "NN"):
        return session_map
    if first_session in ("SR", "SD"):
        return crossover_map
    raise ValueError(f"Unexpected first_session value: {first_session}")

subject_session_maps = {
    row["participant_id"]: build_session_map(
        row["participant_id"],
        first_session=row["first_session"] if has_first_session else None,
        group=row.get("group"),
    )
    for _, row in participants.iterrows()
}

if dataset != "ntnu-bids" and not has_first_session:
    print("No 'first_session' column found, applying session_map to all subjects.")

print(f"Built session maps for {len(subject_session_maps)} subjects. Sample of 5:")
for k in list(subject_session_maps)[:5]:
    print(k, subject_session_maps[k])

# Identify conditions
condition_map = {
    "CL": "control",
    "BL": "baseline",
    "SD": "sleep deprivation",
    "SR": "sleep restriction",
    "RC": "recovery sleep"
}

No 'first_session' column found, applying session_map to all subjects.
Built session maps for 31 subjects. Sample of 5:
sub-002 {'ses-bl': 'ses-T1BL', 'ses-sd': 'ses-T2SD', 'ses-rc': 'ses-T3RC'}
sub-003 {'ses-bl': 'ses-T1BL', 'ses-sd': 'ses-T2SD', 'ses-rc': 'ses-T3RC'}
sub-004 {'ses-bl': 'ses-T1BL', 'ses-sd': 'ses-T2SD', 'ses-rc': 'ses-T3RC'}
sub-006 {'ses-bl': 'ses-T1BL', 'ses-sd': 'ses-T2SD', 'ses-rc': 'ses-T3RC'}
sub-007 {'ses-bl': 'ses-T1BL', 'ses-sd': 'ses-T2SD', 'ses-rc': 'ses-T3RC'}


Scan the dataset for old session labels:

In [9]:
# Collect all items with any old session label
old_labels = set(session_map) | set(crossover_map) | set(control_map)

def exact(label):
    return re.compile(re.escape(label) + r"(?![A-Za-z0-9])")

def relabel(text, smap):
    pat = re.compile("(" + "|".join(map(re.escape, smap)) + r")(?![A-Za-z0-9])")
    return pat.sub(lambda m: smap[m.group(1)], text)

all_items = []
label_counter = Counter()
for item in bids_root.rglob("*"):
    if "sourcedata" in item.parts:  # Ignore sourcedata folder
        continue
    item_str = str(item)
    matched = [old_ses for old_ses in old_labels if exact(old_ses).search(item_str)]
    if matched:
        all_items.append(item)
        label_counter.update(matched)

print(f"Found {len(all_items)} items containing old session labels.")
print("Occurrences of old session labels:")
for label, count in label_counter.items():
    print(f"{label}: {count} items")

Found 5494 items containing old session labels.
Occurrences of old session labels:
ses-sd9: 2747 items
ses-b9: 2747 items


Give a sample of files to be renamed:

In [10]:
# Helper function for matching subject IDs
def get_subject_id(path):
    match = re.search(r"(sub-[a-zA-Z0-9]+)", str(path))
    return match.group(1) if match else None

preview_renames = []
no_subject_generic = []

for item in all_items:
    subj = get_subject_id(item)
    if subj is not None and subj in subject_session_maps:
        ses_map = subject_session_maps[subj]
    else:
        ses_map = session_map
        no_subject_generic.append(item)

    new_path = Path(relabel(str(item), ses_map))
    preview_renames.append((item, new_path))

# Preview first 10
print("Sample of renamed files/folders (first 10):")
for old, new in preview_renames[:10]:
    print(f"{old} -> {new}")

if no_subject_generic:
    print(f"\n{len(no_subject_generic)} items had no subject match — used generic session_map:")
    generic_renames = {old: new for old, new in preview_renames if old in no_subject_generic}
    for s in no_subject_generic[:10]:
        print(f"{s} -> {generic_renames[s]}")

Sample of renamed files/folders (first 10):
/data/project/sleep_ENIGMA_glymphatic/enigma-sd/singapore-effo2-bids/derivatives/freesurfer/ses-sd9 -> /data/project/sleep_ENIGMA_glymphatic/enigma-sd/singapore-effo2-bids/derivatives/freesurfer/ses-T2SD
/data/project/sleep_ENIGMA_glymphatic/enigma-sd/singapore-effo2-bids/derivatives/freesurfer/ses-b9 -> /data/project/sleep_ENIGMA_glymphatic/enigma-sd/singapore-effo2-bids/derivatives/freesurfer/ses-T1BL
/data/project/sleep_ENIGMA_glymphatic/enigma-sd/singapore-effo2-bids/derivatives/freesurfer/ses-sd9/sub-52 -> /data/project/sleep_ENIGMA_glymphatic/enigma-sd/singapore-effo2-bids/derivatives/freesurfer/ses-T2SD/sub-52
/data/project/sleep_ENIGMA_glymphatic/enigma-sd/singapore-effo2-bids/derivatives/freesurfer/ses-sd9/fsaverage -> /data/project/sleep_ENIGMA_glymphatic/enigma-sd/singapore-effo2-bids/derivatives/freesurfer/ses-T2SD/fsaverage
/data/project/sleep_ENIGMA_glymphatic/enigma-sd/singapore-effo2-bids/derivatives/freesurfer/ses-sd9/sub-32 

Rename files:

In [11]:
# Rename files first
renamed_count = 0
errors = []
for old_path, new_path in preview_renames:
    if old_path == new_path:
        continue
    if old_path.is_file():
        try:
            new_path.parent.mkdir(parents=True, exist_ok=True)
            old_path.rename(new_path)
            renamed_count += 1
        except Exception as e:
            errors.append((old_path, new_path, str(e)))
print(f"Renamed {renamed_count} files.")
if errors:
    print(f"{len(errors)} errors occurred:")
    for old, new, err in errors[:10]:
        print(f"{old} -> {new}: {err}")

Renamed 4891 files.


Then rename directories using merging:

In [12]:
def safe_rename_dir(src, dst):
    if not dst.exists():
        src.rename(dst)
    else:
        # Merge contents
        for item in src.iterdir():
            target = dst / item.name
            if item.is_dir():
                safe_rename_dir(item, target)
            else:
                if target.exists():
                    # If file exists, overwrite
                    target.unlink()
                shutil.move(str(item), str(target))
        # Remove the now-empty source directory
        src.rmdir()

# Collect directories containing any old session label (excluding sourcedata)
all_dirs = [
    d for d in bids_root.rglob("*")
    if d.is_dir()
    and "sourcedata" not in d.parts
    and any(exact(old).search(str(d)) for old in old_labels)
]

# Deepest first
all_dirs = sorted(all_dirs, key=lambda x: len(str(x)), reverse=True)

renamed_dir_count = 0
dir_errors = []
for dir_path in all_dirs:
    subj = get_subject_id(dir_path)
    if subj is not None and subj in subject_session_maps:
        ses_map = subject_session_maps[subj]
    else:
        ses_map = session_map

    new_dir_path = Path(relabel(str(dir_path), ses_map))
    if new_dir_path == dir_path:
        continue

    try:
        new_dir_path.parent.mkdir(parents=True, exist_ok=True)
        safe_rename_dir(dir_path, new_dir_path)
        renamed_dir_count += 1
    except Exception as e:
        dir_errors.append((dir_path, new_dir_path, str(e)))

print(f"Renamed {renamed_dir_count} directories (out of {len(all_dirs)} found).")
if dir_errors:
    print(f"{len(dir_errors)} errors occurred:")
    for old, new, err in dir_errors[:10]:
        print(f"{old} -> {new}: {err}")

Renamed 568 directories (out of 568 found).


Create sessions.tsv:

In [13]:
def infer_condition(session_label):
    for key, val in condition_map.items():
        if key.lower() in session_label.lower():
            return val
    return "unknown"

def extract_visit(session_label):
    match = re.search(r'(T\d+)', session_label, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    return "n/a"

def extract_date_time(json_file):
    date, time = "n/a", "n/a"
    try:
        with open(json_file, "r") as f:
            meta = json.load(f)
        raw_date = meta.get("SeriesDate") or meta.get("AcquisitionDate") or meta.get("StudyDate")
        if raw_date:
            raw_date = str(raw_date)
            if "-" in raw_date:
                date = raw_date
            else:
                date = f"{raw_date[:4]}-{raw_date[4:6]}-{raw_date[6:8]}"
        raw_time = meta.get("SeriesTime") or meta.get("AcquisitionTime") or meta.get("StudyTime")
        if raw_time:
            raw_time = str(raw_time).split(".")[0]
            if ":" in raw_time:
                time = raw_time
            else:
                time = f"{raw_time[:2]}:{raw_time[2:4]}:{raw_time[4:6]}"
    except Exception:
        pass
    return date, time

# Iterate over subjects
subjects = sorted([p for p in bids_root.glob("sub-*") if p.is_dir()])
out_files = []

for sub in subjects:
    participant_id = sub.name
    sessions = sorted([s for s in sub.glob("ses-*") if s.is_dir()])
    rows = []
    for ses in sessions:
        session_id = ses.name
        visit = extract_visit(session_id)
        anat_jsons = list((ses / "anat").glob("*T1w.json")) if (ses / "anat").exists() else []
        json_file = anat_jsons[0] if anat_jsons else None
        date, time = extract_date_time(json_file) if json_file else ("n/a", "n/a")
        rows.append({
            "participant_id": participant_id,
            "session_id": session_id,
            "dicom_id": "n/a",
            "visit": visit,
            "condition": infer_condition(session_id),
            "date": date,
            "time": time
        })
    df = pd.DataFrame(rows)
    out_file = sub / f"{participant_id}_sessions.tsv"

    # Unlock if it already exists (annexed symlink) before writing
    if out_file.exists():
        !datalad unlock "{out_file}"

    df.to_csv(out_file, sep="\t", index=False)
    out_files.append(str(out_file))
    print(f"Saved: {out_file.name}")

print(f"\nWrote {len(out_files)} sessions.tsv files. Committing batch...")

Saved: sub-32_sessions.tsv
Saved: sub-37_sessions.tsv
Saved: sub-43_sessions.tsv
Saved: sub-52_sessions.tsv

Wrote 4 sessions.tsv files. Committing batch...


Sanity check:

In [7]:
df

,participant_id,session_id,dicom_id,visit,condition,date,time
0,sub-61,ses-T1BL,n/a,T1,baseline,n/a,08:31:53
1,sub-61,ses-T2SD,n/a,T2,sleep deprivation,n/a,08:31:53
